# Data vs. model occurrence predictions (Rogers 2025, Sec. 4)

Compares the matched young-planet sample against the **gas dwarf** (H2/He, XUV photoevaporation) and **water world** (steam, no escape) occurrence predictions, in two age bins matched to where the sample actually lives: **<=100 Myr** (diagnostic) and **100-700 Myr** (bulk).

Model and data matching mirror `rogers2025_gasdwarf_waterworld.ipynb`; the heavy lifting lives in `data_vs_model.py`. Synthesis takes ~60-90 s.

> Caveats baked into the comparison: raw detections (not occurrence-corrected; young-star surveys favour large planets), main-sequence Teq (no pre-main-sequence dimming), and a simplified model (constant opacity, log-normal core/envelope distributions, N=2000). Results are indicative, not a calibrated Bayes factor.

In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import data_vs_model as dvm

# load project matplotlib config (rcParams.txt)
with open('../rcParams.txt') as f:
    for name, value in csv.reader(f):
        if name == 'name':
            continue
        try:
            value = int(value)
        except ValueError:
            try:
                value = float(value)
            except ValueError:
                pass
        try:
            plt.rcParams[name] = value
        except Exception:
            pass
print('setup done')

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (60)

setup done


In [2]:
dp = dvm.load_planets()
pops = dvm.synthesize(seed=42)
results = dvm.compare(pops, dp)
results

/mnt_ut2/raid_ut2/home/jerome/github/research/project/wakai/radius_evolution/data_vs_model.py:59: RuntimeWarning: divide by zero encountered in scalar divide
  A = np.sum(_GLw * 4 * np.pi * r * r * f) * jac; rho = Menv / A
/mnt_ut2/raid_ut2/home/jerome/github/research/project/wakai/radius_evolution/data_vs_model.py:63: RuntimeWarning: invalid value encountered in scalar multiply
  Ep = rho * B + Mc * kB * Tc / (mu_c * mH * (gamma_c - 1))
/mnt_ut2/raid_ut2/home/jerome/github/research/project/wakai/radius_evolution/data_vs_model.py:87: RuntimeWarning: invalid value encountered in scalar multiply
  rr = rho * np.exp((G * Mc / cs2) * (1 / r - 1 / Rr)); H = cs2 * r * r / (G * Mc)


,bin,N_data,med_R_data,med_R_gd,med_R_ww,frac_gt3_data,frac_gt3_gd,frac_gt3_ww,dlnL_gd_minus_ww,favored
0,<=100 Myr,22,4.85,2.75,2.22,0.73,0.35,0.04,466.21,gas dwarf
1,100-700 Myr,53,2.39,2.07,2.17,0.19,0.00,0.01,-358.57,water world


## What the empirical data tell us

| Age bin | N | median R (data) | gas dwarf | water world | frac >3 R_E (data/gd/ww) | favoured |
|---|---|---|---|---|---|---|
| <=100 Myr | 22 | 4.85 | 2.75 | 2.22 | 0.73 / 0.35 / 0.04 | **gas dwarf** |
| 100-700 Myr | 53 | 2.39 | 2.07 | 2.17 | 0.19 / 0.00 / 0.01 | water world (weak) |

- **The signal is in the youngest bin.** At <=100 Myr the planets are strongly inflated (median 4.85 R_E, 73% above 3 R_E). A steam envelope essentially cannot reach those radii (4%); only the low-mu H2/He gas-dwarf model makes a real inflated tail (35%). The data favour **H2/He envelopes** - the degeneracy-breaking leverage of young transiting planets.
- **By a few hundred Myr the discriminating power is gone.** Both models converge near 2.1 R_E; the mild 'water world' preference is not meaningful (the gas-dwarf model has over-stripped, predicting too few large planets, while the data keep a 19% tail neither model reproduces). Read this bin as uninformative - exactly the convergence Rogers (2025) predicts.

Numbers above are from a representative seed-fixed run; the cell output is authoritative.

In [ ]:
# marginal radius distributions: data vs the two models, per age bin
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
rbins = np.linspace(0, 10, 30)
for ax, (label, epoch, lo, hi) in zip(axes, dvm.AGE_BINS):
    sub = dp[(dp.amid >= lo) & (dp.amid < hi) & (dp.pl_orbper < 100) & (dp.pl_rade < 10)]
    ax.hist(sub.pl_rade, bins=rbins, density=True, color='k', alpha=0.4,
            label=f'data (N={len(sub)})')
    ax.hist(pops[('gd', epoch)][1], bins=rbins, density=True, histtype='step',
            color='C2', lw=2.5, label='gas dwarf')
    ax.hist(pops[('ww', epoch)][1], bins=rbins, density=True, histtype='step',
            color='C4', lw=2.5, label='water world')
    ax.axvline(1.8, color='gray', ls=':', lw=1)
    ax.set_xlabel(r'Planet radius [R$_\oplus$]')
    ax.set_title(label)
    ax.legend(frameon=False, fontsize=11)
axes[0].set_ylabel('normalised density')
fig.tight_layout()
fig.savefig('../figures/rogers2025_data_vs_model_radius_dist.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
dvm.targets_table()

## 'Our Work' targets

| Target | P [d] | R [R_E] | valley R | R / valley | role |
|---|---|---|---|---|---|
| TIC*1445 b | 3.27 | 5.7 | 2.11 | 2.7 | inflated Neptune - gas-dwarf-favouring |
| TOI-6715 b | 2.86 | 4.75 | 2.13 | 2.2 | inflated sub-Neptune - gas-dwarf-favouring |
| TOI-5973 | 3.16 | 24.3 | 2.11 | 11.5 | hot Jupiter - off-diagram (R>10) |
| TOI-6940 b | 0.31 | 1.4 | 2.61 | 0.54 | rocky USP - off-panel (P<1) |
| TOI-5169 b | 8.04 | 2.26 | 1.94 | 1.16 | small sub-Neptune near gap |
| TOI-5734 | 6.18 | 2.10 | 1.99 | 1.06 | sits on the radius valley |

- **TIC*1445 b and TOI-6715 b** are the decisive targets: inflated Neptunes well above the valley, in the region only the gas-dwarf model populates at young ages.
- **TOI-5169 b, TOI-5734, TOI-6940 b** are small / valley-straddling - consistent with stripped or rocky outcomes; weakly discriminating.
- **TOI-5973** (2.17 R_jup ~ 24 R_E) is an inflated giant, not a sub-Neptune; it falls outside the 1-10 R_E panel and is not part of this comparison.

Plot-window flags (`in_panel`) and valley ratios are computed in `targets_table()`. Two targets fall outside the 1-100 d x 1-10 R_E window (TOI-5973, TOI-6940 b) and will not render on the period-radius panels.